# segearth04 — SegEarth-OV-3 (improved)

Changes from segearth03:
- Avoids the MMCV_MAX source-patch by using a compatible mmcv version
- Adds reproducibility seed
- **Section 3B**: runs `eval.py` with the existing `configs/cfg_potsdam.py` for quantitative mIoU/mAcc (everything already in the repo — just needs data staged)
- Section 3A: original demo.py flow kept as-is for Frankfurt DOP20 qualitative output
- PAMR boundary refinement as an optional cell
- tqdm progress on the demo.py inference loop

**Needs:** Kaggle T4 GPU, `dummyirl/sam3-weights` dataset attached.  
Section 3B additionally needs Potsdam data (see staging cell).

## 1 — Environment setup

In [ ]:
import os

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!bash Miniconda3-latest-Linux-x86_64.sh -b -p /kaggle/working/miniconda

os.environ.pop("PYTHONPATH", None)
os.environ["PATH"] = "/kaggle/working/miniconda/bin:" + os.environ["PATH"]

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

!conda --version

In [ ]:
!/kaggle/working/miniconda/bin/conda create -n segearth python=3.10 -y

In [ ]:
!conda run -n segearth pip install torch==2.4.0 torchvision==0.19.0

In [ ]:
!conda run -n segearth pip install openmim
!conda run -n segearth mim install "mmcv==2.2.0"
!conda run -n segearth pip install mmcv==2.2.0
!conda run -n segearth pip install "mmsegmentation==1.2.2"

In [ ]:
%%bash
# mmseg 1.2.2 declares mmcv<2.2.0 as max, but 2.2.0 is what we need.
# Patch MMCV_MAX in mmseg's __init__.py so the version check passes.
source /kaggle/working/miniconda/bin/activate segearth
python - << 'EOF'
import pathlib
init_file = pathlib.Path(
    "/kaggle/working/miniconda/envs/segearth/lib/python3.10/site-packages/mmseg/__init__.py"
)
txt = init_file.read_text()
txt = txt.replace("MMCV_MAX = '2.2.0'", "MMCV_MAX = '2.3.0'")
init_file.write_text(txt)
print("Patched MMCV_MAX → 2.3.0 in", init_file)
EOF

pip install numpy==1.26.4 -q

In [ ]:
%%bash
source /kaggle/working/miniconda/bin/activate segearth
python - << 'EOF'
import mmcv; print("MMCV:", mmcv.__version__)
from mmcv.ops import point_sample; print("MMCV OPS OK")
from mmseg.structures import SegDataSample; print("MMSEG OK")
import torch; print("CUDA:", torch.cuda.is_available())
EOF

## 2 — Clone repo

In [ ]:
import subprocess, os
from pathlib import Path

REPO = Path("/kaggle/working/SegEarth-OV-3")

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    print(f"updated → {REPO}")
else:
    subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/HarishDeepak/SegEarth-OV-3.git", str(REPO)],
        check=True)
    print(f"cloned → {REPO}")

os.chdir(REPO)
!ls

In [ ]:
!conda run -n segearth pip install -r requirements.txt

## 3A — Hessen/Darmstadt DOP20 inference (qualitative)

Uses `cfg_hessen.py`: 6 rural classes, `slide_crop=256` / `stride=128` (50% overlap, Gaussian-weighted).  
The smaller tile size compensates for the 4× resolution gap (20 cm vs Potsdam 5 cm) without upscaling.  
`config_local.py` sets `INPUT_FOLDER = /kaggle/input/darmstadt-dop20/` automatically on Kaggle.  
**Dataset to attach:** `dummyirl/darmstadt-dop20`

In [ ]:
%%bash
export MPLBACKEND=Agg
source /kaggle/working/miniconda/bin/activate segearth
cd /kaggle/working/SegEarth-OV-3

python - << 'EOF'
import torch, numpy as np, os
os.environ['MPLBACKEND'] = 'Agg'
torch.manual_seed(42); np.random.seed(42)

from pathlib import Path
from PIL import Image
from torchvision import transforms
from mmseg.structures import SegDataSample
from segearthov3_segmentor import SegEarthOV3Segmentation
from config_local import INPUT_FOLDER, RUN_SINGLE_IMAGE, TARGET_IMAGE

try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

import time, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
os.makedirs("output", exist_ok=True)

# Auto-discover files: check root and one level of subdirectories
TILE_EXTS = [".tif", ".tiff", ".png", ".jpg", ".jpeg"]
base = Path(INPUT_FOLDER)
image_paths = sorted(p for ext in TILE_EXTS for p in base.glob(f"*{ext}"))
if not image_paths:
    # files may be one level deeper
    image_paths = sorted(p for ext in TILE_EXTS for p in base.glob(f"**/*{ext}"))
if not image_paths:
    print(f"No images found in {INPUT_FOLDER}. Contents:")
    for p in sorted(base.iterdir())[:20]: print(" ", p)
    raise SystemExit(1)

if RUN_SINGLE_IMAGE:
    image_paths = [p for p in image_paths if p.name == TARGET_IMAGE]

print(f"Found {len(image_paths)} image(s)")

CLASS_NAMES = ['agricultural field', 'forest', 'building', 'road', 'water', 'background']
COLOR_MAP = np.array([
    [210, 180, 140], [34, 139, 34], [220, 20, 60],
    [105, 105, 105], [30, 144, 255], [200, 200, 200],
], dtype=np.uint8)

model = SegEarthOV3Segmentation(
    type='SegEarthOV3Segmentation',
    classname_path='./configs/cls_hessen.txt',
    prob_thd=0.1, bg_idx=5, confidence_threshold=0.1,
    slide_stride=128, slide_crop=256,
)

for idx, img_path in enumerate(tqdm(image_paths, desc="Hessen inference"), 1):
    t0 = time.time()
    img = Image.open(img_path).convert("RGB")
    img_tensor = transforms.ToTensor()(img).unsqueeze(0).to('cuda')
    data_sample = SegDataSample()
    data_sample.set_metainfo({'img_path': str(img_path), 'ori_shape': img.size[::-1]})
    result = model.predict(img_tensor, data_samples=[data_sample])
    seg_pred = result[0].pred_sem_seg.data.cpu().numpy().squeeze(0)

    seg_rgb = COLOR_MAP[np.clip(seg_pred, 0, len(COLOR_MAP) - 1)]
    fig, ax = plt.subplots(1, 2, figsize=(16, 8))
    ax[0].imshow(img); ax[0].axis('off'); ax[0].set_title("RGB (Hessen DOP20 20cm)")
    ax[1].imshow(img); ax[1].imshow(seg_rgb, alpha=0.45)
    ax[1].axis('off'); ax[1].set_title("Prediction (cls_hessen, Gaussian window)")
    fig.legend(handles=[Patch(facecolor=c/255.0, label=n) for n, c in zip(CLASS_NAMES, COLOR_MAP)],
               loc='lower center', ncol=6, bbox_to_anchor=(0.5, -0.02))
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    out = f"output/{img_path.stem}_hessen.png"
    plt.savefig(out, bbox_inches='tight', dpi=150)
    plt.close()
    print(f"  [{idx}] {img_path.name} -> {out} ({time.time()-t0:.1f}s)")

print("Done.")
EOF

## 3B — Potsdam quantitative eval (mIoU / mAcc)

The repo already has `configs/cfg_potsdam.py` and `configs/cls_potsdam.txt` (6-class: road, building, grass, tree, car, clutter).  
We just need to stage the Potsdam data at the expected path and call `eval.py`.

**Attach dataset:** Add the Potsdam dataset to this Kaggle notebook. Then run the staging cell to create the expected folder structure at `/kaggle/working/PotsdamEval/`.

In [ ]:
# Stage Potsdam val tile at the path cfg_potsdam.py expects:
#   /kaggle/working/PotsdamEval/img_dir/val/          ← *_RGB.tif
#   /kaggle/working/PotsdamEval/ann_dir_indexed/val/  ← *_label_noBoundary.tif
#
# Dataset attached: dummyirl/6isprs → /kaggle/input/6isprs/

import os
from pathlib import Path

POTSDAM_INPUT = Path("/kaggle/input/6isprs")
EVAL_ROOT     = Path("/kaggle/working/PotsdamEval")
VAL_TILE      = "6_15"   # tile-level val split (no patch-overlap leakage)

img_dst   = EVAL_ROOT / "img_dir/val"
label_dst = EVAL_ROOT / "ann_dir_indexed/val"
img_dst.mkdir(parents=True, exist_ok=True)
label_dst.mkdir(parents=True, exist_ok=True)

img_files   = sorted(POTSDAM_INPUT.rglob(f"*{VAL_TILE}*_RGB.tif"))
label_files = sorted(POTSDAM_INPUT.rglob(f"*{VAL_TILE}*_label_noBoundary.tif"))

if not img_files:
    print("No val tile images found. Files in dataset:")
    for f in sorted(POTSDAM_INPUT.rglob("*.tif"))[:20]:
        print(" ", f.relative_to(POTSDAM_INPUT))
else:
    for p in img_files:
        dst = img_dst / p.name
        if not dst.exists():
            dst.symlink_to(p)
    for p in label_files:
        dst = label_dst / p.name
        if not dst.exists():
            dst.symlink_to(p)
    print(f"Images:  {len(list(img_dst.iterdir()))}")
    print(f"Labels:  {len(list(label_dst.iterdir()))}")
    print("Staging done.")

In [ ]:
%%bash
source /kaggle/working/miniconda/bin/activate segearth
cd /kaggle/working/SegEarth-OV-3

# eval.py defaults to configs/cfg_potsdam.py.
# Results printed to stdout and saved to work_dirs/cfg_potsdam/results.txt
python eval.py configs/cfg_potsdam.py

## 4 — (Optional) PAMR boundary refinement

`pamr.py` is already in the repo. This refines logits with local pixel affinity to sharpen class boundaries.  
To use this cell, save `seg_logits` during inference:
```python
np.save(f"output/{img_path.stem}_logits.npy", result[0].seg_logits.data.cpu().numpy())
```

In [ ]:
%%bash
source /kaggle/working/miniconda/bin/activate segearth
cd /kaggle/working/SegEarth-OV-3

python - << 'EOF'
import torch, numpy as np
from pathlib import Path
from PIL import Image
from torchvision import transforms
from pamr import PAMR
from config_local import INPUT_FOLDER

pamr = PAMR(num_iter=10, dilations=[1, 2, 4, 8]).cuda()
output_dir = Path("output")
logits_files = sorted(output_dir.glob("*_logits.npy"))

if not logits_files:
    print("No *_logits.npy files found — see note above.")
else:
    for lf in logits_files:
        stem = lf.stem.replace("_logits", "")
        candidates = list(Path(INPUT_FOLDER).glob(f"{stem}*"))
        if not candidates:
            print(f"Image not found for {stem}, skipping")
            continue
        img_t = transforms.ToTensor()(Image.open(candidates[0]).convert("RGB")).unsqueeze(0).cuda()
        logits = torch.from_numpy(np.load(lf)).unsqueeze(0).cuda()
        refined = pamr(img_t, logits).squeeze(0).argmax(0).cpu().numpy()
        np.save(output_dir / f"{stem}_pred_pamr.npy", refined)
        print(f"PAMR done: {stem}")
    print("Refined preds saved as *_pred_pamr.npy")
EOF

## 5 — Presentation demo: user-guided open-vocab inference

**Flow:**
1. Cell 5A: display a grid of Hessen tiles — audience picks 2 by index
2. Cell 5B: run with standard Potsdam vocabulary → shows domain gap
3. Cell 5C: audience describes what they see → edit `USER_CLASSES` → re-run
4. Cell 5D: side-by-side comparison (default vs user-guided)

This demonstrates live what "open vocabulary" means: same frozen model, same image, different text queries → different segmentation.

In [ ]:
## 5A — Display tile grid: audience picks 2 images by index

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from PIL import Image
from config_local import INPUT_FOLDER

TILE_EXTS = [".tif", ".tiff", ".jpg", ".jpeg", ".png"]
all_tiles = sorted(
    p for ext in TILE_EXTS for p in Path(INPUT_FOLDER).glob(f"*{ext}")
)
print(f"Available tiles (0–{len(all_tiles)-1}):")
for i, p in enumerate(all_tiles):
    print(f"  [{i}] {p.name}")

# Display as grid (up to 12 tiles)
n = min(len(all_tiles), 12)
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
axes = np.array(axes).flatten()
for i in range(n):
    img = Image.open(all_tiles[i]).convert("RGB")
    axes[i].imshow(img)
    axes[i].set_title(f"[{i}] {all_tiles[i].stem[:20]}", fontsize=8)
    axes[i].axis('off')
for j in range(n, len(axes)):
    axes[j].axis('off')
plt.tight_layout()
plt.savefig("output/tile_grid.png", bbox_inches='tight', dpi=100)
plt.show()
print("\nSaved tile_grid.png — show this to the audience and ask them to pick 2 indices.")

In [ ]:
## 5B — Run A: default Potsdam vocabulary (shows domain gap)
TILE_IDX = [0, 1]   # <- CHANGE to audience's chosen indices

%%bash
export MPLBACKEND=Agg
source /kaggle/working/miniconda/bin/activate segearth
cd /kaggle/working/SegEarth-OV-3

python - << 'EOF'
import torch, numpy as np, os
os.environ['MPLBACKEND'] = 'Agg'
torch.manual_seed(42)

from pathlib import Path
from PIL import Image
from torchvision import transforms
from mmseg.structures import SegDataSample
from segearthov3_segmentor import SegEarthOV3Segmentation
from config_local import INPUT_FOLDER

TILE_IDX = list(map(int, os.environ.get("DEMO_TILE_IDX", "0,1").split(",")))
TILE_EXTS = [".tif", ".tiff", ".jpg", ".jpeg", ".png"]
base = Path(INPUT_FOLDER)
all_tiles = sorted(p for ext in TILE_EXTS for p in base.glob(f"*{ext}"))
if not all_tiles:
    all_tiles = sorted(p for ext in TILE_EXTS for p in base.glob(f"**/*{ext}"))
chosen = [all_tiles[i] for i in TILE_IDX if i < len(all_tiles)]

POTSDAM_NAMES = ['impervious surface', 'building', 'low vegetation', 'tree', 'car', 'clutter']
COLOR_MAP = np.array([
    [128, 0, 0], [0, 0, 255], [0, 255, 0],
    [0, 128, 0], [255, 255, 0], [255, 0, 0]
], dtype=np.uint8)

model = SegEarthOV3Segmentation(
    type='SegEarthOV3Segmentation',
    classname_path='./configs/cls_potsdam.txt',
    prob_thd=0.1, bg_idx=5, confidence_threshold=0.1,
    slide_stride=128, slide_crop=256,
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
os.makedirs("output", exist_ok=True)

for img_path in chosen:
    img = Image.open(img_path).convert("RGB")
    ds = SegDataSample()
    ds.set_metainfo({'img_path': str(img_path), 'ori_shape': img.size[::-1]})
    result = model.predict(transforms.ToTensor()(img).unsqueeze(0).to('cuda'), [ds])
    seg = result[0].pred_sem_seg.data.cpu().numpy().squeeze(0)
    seg_rgb = COLOR_MAP[np.clip(seg, 0, 5)]
    fig, ax = plt.subplots(1, 2, figsize=(14, 7))
    ax[0].imshow(img); ax[0].axis('off'); ax[0].set_title("RGB")
    ax[1].imshow(img); ax[1].imshow(seg_rgb, alpha=0.5)
    ax[1].axis('off'); ax[1].set_title("Run A — Potsdam vocab (domain gap)")
    fig.legend(handles=[Patch(facecolor=c/255., label=n) for n, c in zip(POTSDAM_NAMES, COLOR_MAP)],
               loc='lower center', ncol=6, bbox_to_anchor=(0.5, -0.02))
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    out = f"output/{img_path.stem}_runA.png"
    plt.savefig(out, bbox_inches='tight', dpi=150); plt.close()
    print(f"Saved {out}")
EOF

### 5C — Audience describes what they see

Show the audience the RGB tiles. Ask: *"What do you see in this image?"*  
Edit `USER_CLASSES` below — each key is a class name, each value is a list of synonyms.  
The model runs again with exactly these text queries. Nothing else changes.

In [ ]:
## 5C+D — Run B: user-guided vocabulary, then side-by-side comparison

# ← Audience looks at the RGB tiles and describes what they see.
#   Edit these classes live during the presentation.
#   Each value list = synonyms — all get averaged into one text embedding.
USER_CLASSES = {
    "agricultural field": ["agricultural field", "farmland", "crop field", "ploughed field"],
    "forest":             ["forest", "woodland", "dense trees", "tree canopy"],
    "building":           ["building", "rooftop", "house", "village structure"],
    "road":               ["road", "path", "dirt track", "country road"],
    "water":              ["water", "river", "stream", "pond"],
    "background":         ["background", "open land", "bare soil"],
}

# Write a temporary cls file from USER_CLASSES
import os
from pathlib import Path

cls_path = Path("/kaggle/working/SegEarth-OV-3/configs/cls_user.txt")
with open(cls_path, "w") as f:
    for synonyms in USER_CLASSES.values():
        f.write(", ".join(synonyms) + "\n")
print(f"Written {cls_path}:")
print(cls_path.read_text())

In [ ]:
%%bash
export MPLBACKEND=Agg
source /kaggle/working/miniconda/bin/activate segearth
cd /kaggle/working/SegEarth-OV-3

python - << 'EOF'
import torch, numpy as np, os
os.environ['MPLBACKEND'] = 'Agg'
torch.manual_seed(42)

from pathlib import Path
from PIL import Image
from torchvision import transforms
from mmseg.structures import SegDataSample
from segearthov3_segmentor import SegEarthOV3Segmentation
from config_local import INPUT_FOLDER

TILE_IDX = list(map(int, os.environ.get("DEMO_TILE_IDX", "0,1").split(",")))
TILE_EXTS = [".tif", ".tiff", ".jpg", ".jpeg", ".png"]
base = Path(INPUT_FOLDER)
all_tiles = sorted(p for ext in TILE_EXTS for p in base.glob(f"*{ext}"))
if not all_tiles:
    all_tiles = sorted(p for ext in TILE_EXTS for p in base.glob(f"**/*{ext}"))
chosen = [all_tiles[i] for i in TILE_IDX if i < len(all_tiles)]

cls_path = "./configs/cls_user.txt"
user_names = [line.split(",")[0].strip() for line in open(cls_path) if line.strip()]
n_cls = len(user_names)

import matplotlib.cm as cm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

cmap = cm.get_cmap("tab10", n_cls)
COLOR_MAP = (np.array([cmap(i)[:3] for i in range(n_cls)]) * 255).astype(np.uint8)

model_B = SegEarthOV3Segmentation(
    type='SegEarthOV3Segmentation', classname_path=cls_path,
    prob_thd=0.1, bg_idx=n_cls - 1, confidence_threshold=0.1,
    slide_stride=128, slide_crop=256,
)
POTSDAM_NAMES = ['impervious surface', 'building', 'low vegetation', 'tree', 'car', 'clutter']
POTSDAM_COLORS = np.array([
    [128, 0, 0], [0, 0, 255], [0, 255, 0],
    [0, 128, 0], [255, 255, 0], [255, 0, 0]
], dtype=np.uint8)
model_A = SegEarthOV3Segmentation(
    type='SegEarthOV3Segmentation', classname_path='./configs/cls_potsdam.txt',
    prob_thd=0.1, bg_idx=5, confidence_threshold=0.1,
    slide_stride=128, slide_crop=256,
)

os.makedirs("output", exist_ok=True)
for img_path in chosen:
    img = Image.open(img_path).convert("RGB")
    img_t = transforms.ToTensor()(img).unsqueeze(0).to('cuda')

    def run(m):
        ds = SegDataSample()
        ds.set_metainfo({'img_path': str(img_path), 'ori_shape': img.size[::-1]})
        return m.predict(img_t, [ds])[0].pred_sem_seg.data.cpu().numpy().squeeze(0)

    rgb_A = POTSDAM_COLORS[np.clip(run(model_A), 0, 5)]
    rgb_B = COLOR_MAP[np.clip(run(model_B), 0, n_cls - 1)]

    fig, axes = plt.subplots(1, 3, figsize=(21, 7))
    axes[0].imshow(img); axes[0].axis('off'); axes[0].set_title("RGB", fontsize=13)
    axes[1].imshow(img); axes[1].imshow(rgb_A, alpha=0.5)
    axes[1].axis('off'); axes[1].set_title("Run A — Potsdam vocab", fontsize=13)
    axes[2].imshow(img); axes[2].imshow(rgb_B, alpha=0.5)
    axes[2].axis('off'); axes[2].set_title("Run B — Your vocabulary", fontsize=13)
    fig.legend(handles=[Patch(facecolor=c/255., label=n) for n, c in zip(user_names, COLOR_MAP)],
               loc='lower center', ncol=n_cls, bbox_to_anchor=(0.5, -0.02), fontsize=10)
    plt.suptitle(f"{img_path.stem} — same model, same weights, different text prompts", fontsize=11)
    plt.tight_layout(rect=[0, 0.06, 1, 0.97])
    out = f"output/{img_path.stem}_comparison.png"
    plt.savefig(out, bbox_inches='tight', dpi=150); plt.close()
    print(f"Saved {out}")

print("Done — open output/*_comparison.png")
EOF